# Synthetic MuSiQue: 30-question review batch

Target **five structurally valid, normalized-exact-unique questions for each of the six hop subtypes**.
This is a diagnostic sample; accepted records still need semantic review.

Select the **capstone** Python kernel. Run setup and definitions, then deliberately enable the generation cell when ready.
**Generation makes paid calls. It is disabled by default.** No calls are needed to reopen saved outputs or review them.

The default cap is 10 requests per subtype, with at most 2 model calls per request: **120 client calls maximum**,
usually 30 calls if every first attempt succeeds. SDK-internal retries are separate. The batch may finish below 30 accepted
records if a subtype reaches its cap. API/programming errors stop the batch and retain available history.


In [55]:
import csv
import hashlib
import html
import json
import sys
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

from IPython.display import HTML, display

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src/mas_sae/data/synthetic").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the repository.")
sys.path.insert(0, str(ROOT / "src"))

from mas_sae.data.synthetic.batch import run_batch
from mas_sae.data.synthetic.generation import PROMPT_VERSION
from mas_sae.data.synthetic.generation_cli import DEFAULT_MODEL_ID, make_client
from mas_sae.data.synthetic.hop_types import HOP_TYPES

MODEL_ID = DEFAULT_MODEL_ID
HOP_TYPES_TO_REVIEW = tuple(HOP_TYPES)


# CONCURRENCY = 64  # Benchmark 1, 4, 8, then 16 in separate runs.
# CONCURRENCY = 128  # Benchmark 1, 4, 8, then 16 in separate runs.
# CONCURRENCY = 256  # Benchmark 1, 4, 8, then 16 in separate runs.
CONCURRENCY = 256  # Benchmark 1, 4, 8, then 16 in separate runs.
# CONCURRENCY = 512  # Benchmark 1, 4, 8, then 16 in separate runs.


TARGET_PER_TYPE = 200
MAX_REQUESTS_PER_TYPE = 300


MAX_ATTEMPTS = 2
MAX_TOKENS = 6000
TEMPERATURE = None  # Omit temperature for Opus.
CONNECT_TIMEOUT = 10
READ_TIMEOUT = 300
SDK_MAX_ATTEMPTS = 3
SEED = 42  # Local sampling/shuffling only; not a model-generation seed.
ORIGINAL_FILES = (
    ROOT / "data/MuSiQue/clean/train.json",
    ROOT / "data/MuSiQue/clean/validation.json",
)
# RUN_DIR = ROOT / "data/MuSiQue/synthetic/runs" / "benchmarks" / (
#     datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
#     + "_review30_" + uuid4().hex[:8]
# )
RUN_DIR = ROOT / "data/MuSiQue/synthetic/runs" / "sample" / (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    + "_review30_" + uuid4().hex[:8]
)
print(f"Model: {MODEL_ID} | prompt: {PROMPT_VERSION}")
print(f"Concurrency: {CONCURRENCY}")
print(f"Target: {TARGET_PER_TYPE * len(HOP_TYPES_TO_REVIEW)} accepted records")
print(f"Maximum client calls: {MAX_REQUESTS_PER_TYPE * len(HOP_TYPES_TO_REVIEW) * MAX_ATTEMPTS}")
print(f"Output: {RUN_DIR}")


Model: us.anthropic.claude-opus-5 | prompt: v3
Concurrency: 256
Target: 1200 accepted records
Maximum client calls: 3600
Output: /home/ubuntu/capstone/fall-2026-group3/data/MuSiQue/synthetic/runs/sample/20261030T173616Z_review30_8cfe1443


In [56]:
import inspect
print(inspect.signature(make_client))

(model_id: str, *, connect_timeout: float = 10, read_timeout: float = 300, sdk_max_attempts: int = 3)


## Shared batch runner — no API calls here

The notebook and CLI both call `synthetic.batch.run_batch`. Up to `CONCURRENCY` workers generate independently;
one coordinator validates, deduplicates, writes files, and updates quotas. Accepted + in-flight counts prevent quota overshoot.
Worker clients are reused and closed at shutdown; each request has its own saved seed and provenance.

The duplicate index covers both original splits and this batch, not other synthetic runs. Outputs are
`candidates.jsonl`, `accepted.jsonl`, `validation.jsonl`, `provenance.jsonl`, `summary.json`, and the SQLite index.
Existing output directories are refused. On a fatal error or interruption, new scheduling stops and already-running
calls are drained and saved before failure is propagated. Shutdown can take time; SDK attempts and socket timeouts are bounded.
There is no automatic resume.


In [57]:
def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]


def run_review_batch(output_dir):
    transport = {
        "connect_timeout": CONNECT_TIMEOUT, "read_timeout": READ_TIMEOUT,
        "sdk_max_attempts": SDK_MAX_ATTEMPTS, "retry_mode": "standard",
    }

    def factory():
        return make_client(
            MODEL_ID, connect_timeout=CONNECT_TIMEOUT,
            read_timeout=READ_TIMEOUT, sdk_max_attempts=SDK_MAX_ATTEMPTS,
        )

    def progress(report):
        state = "accepted" if report["accepted"] else "rejected"
        print(f"{report['request_id']}: {state}", flush=True)

    return run_batch(
        output_dir, original=ORIGINAL_FILES[0], dedup_originals=ORIGINAL_FILES,
        model_id=MODEL_ID, target_per_type=TARGET_PER_TYPE,
        max_requests_per_type=MAX_REQUESTS_PER_TYPE, hop_types=HOP_TYPES_TO_REVIEW,
        concurrency=CONCURRENCY, seed=SEED, max_attempts=MAX_ATTEMPTS,
        max_tokens=MAX_TOKENS, temperature=TEMPERATURE,
        client_factory=factory, transport_settings=transport, progress=progress,
    )


## Generate — paid calls only when enabled

Change `RUN_GENERATION` to `True` and execute this cell deliberately. A new batch refuses an existing `RUN_DIR`.
To generate a separate batch, rerun configuration to obtain a new directory. Reset the switch afterward. Set `CONCURRENCY = 16` in configuration to allow up to 16 active calls.
The default stays at 1 until you choose a concurrency for your account.
If a run fails or is interrupted, load its saved output below; this notebook does not automatically resume or regenerate it.


In [ ]:
RUN_GENERATION = True
if RUN_GENERATION:
    batch_summary = run_review_batch(RUN_DIR)
    print(json.dumps(batch_summary, indent=2))
else:
    print("Generation disabled. No API calls made.")


2hop_request_000042: accepted
3hop1_request_000035: accepted
2hop_request_000020: accepted
2hop_request_000024: accepted
2hop_request_000026: accepted
2hop_request_000030: accepted
2hop_request_000032: accepted
2hop_request_000025: accepted
2hop_request_000002: accepted
2hop_request_000017: accepted
2hop_request_000001: accepted
3hop1_request_000001: accepted
2hop_request_000029: accepted
2hop_request_000039: accepted
3hop1_request_000025: accepted
2hop_request_000033: accepted
2hop_request_000005: accepted
2hop_request_000006: accepted
2hop_request_000031: accepted
2hop_request_000016: accepted
3hop1_request_000012: accepted
2hop_request_000008: accepted
3hop1_request_000007: accepted
3hop1_request_000004: accepted
2hop_request_000022: accepted
2hop_request_000023: accepted
2hop_request_000003: accepted
2hop_request_000004: accepted
2hop_request_000009: accepted
2hop_request_000019: accepted
3hop1_request_000021: accepted
2hop_request_000021: accepted
3hop1_request_000009: accepted
3h

## Load saved outputs — no API calls

For a previous session, replace `REVIEW_RUN_DIR` with its saved directory path.
You can load a partial run after an error. These cells never call generation.


In [ ]:
REVIEW_RUN_DIR = RUN_DIR  # Or: ROOT / "data/MuSiQue/synthetic/runs/<saved-run>"

if not (REVIEW_RUN_DIR / "summary.json").is_file():
    print("No saved batch here yet. Generate it explicitly or set REVIEW_RUN_DIR to an existing run.")
    records, provenance_rows, validation_rows = [], [], []
else:
    saved_summary = json.loads((REVIEW_RUN_DIR / "summary.json").read_text())
    records = read_jsonl(REVIEW_RUN_DIR / "accepted.jsonl") if (REVIEW_RUN_DIR / "accepted.jsonl").exists() else []
    provenance_rows = read_jsonl(REVIEW_RUN_DIR / "provenance.jsonl") if (REVIEW_RUN_DIR / "provenance.jsonl").exists() else []
    validation_rows = read_jsonl(REVIEW_RUN_DIR / "validation.jsonl") if (REVIEW_RUN_DIR / "validation.jsonl").exists() else []
    print(json.dumps(saved_summary, indent=2))
    rows = []
    for hop in HOP_TYPES_TO_REVIEW:
        requested = saved_summary["requests_by_type"].get(hop, 0)
        count = saved_summary["accepted_by_type"].get(hop, 0)
        rate = f"{count / requested:.0%}" if requested else "—"
        rows.append(f"<tr><td>{hop}</td><td>{requested}</td><td>{count}</td><td>{rate}</td></tr>")
    display(HTML("<table><tr><th>Subtype</th><th>Requests</th><th>Accepted</th>"
                 "<th>Acceptance per request</th></tr>" + "".join(rows) + "</table>"))
    failure_counts = Counter(
        error["code"] for row in validation_rows for error in row["errors"]
    )
    print("Request rejection reasons:", dict(failure_counts))
    print("A successful request can include an earlier rejected model response; inspect provenance for those details.")


{
  "status": "completed",
  "fatal_error": null,
  "secondary_errors": [],
  "generated": 1205,
  "accepted": 1200,
  "failed": 0,
  "rejected": 5,
  "cancelled": 0,
  "api_calls": 1211,
  "completed_responses": 1211,
  "input_tokens": 1618796,
  "output_tokens": 2640355,
  "sdk_retry_attempts": 2,
  "accepted_total": 1200,
  "requests_submitted": 1205,
  "results_saved": 1205,
  "concurrency": 512,
  "generated_by_hop_type": {
    "2hop": 202,
    "3hop1": 202,
    "4hop3": 200,
    "3hop2": 200,
    "4hop1": 201,
    "4hop2": 200
  },
  "requests_by_type": {
    "2hop": 202,
    "3hop1": 202,
    "3hop2": 200,
    "4hop1": 201,
    "4hop2": 200,
    "4hop3": 200
  },
  "accepted_by_type": {
    "2hop": 200,
    "3hop1": 200,
    "3hop2": 200,
    "4hop1": 200,
    "4hop2": 200,
    "4hop3": 200
  },
  "model_id": "us.anthropic.claude-opus-5",
  "prompt_version": "v3",
  "max_tokens": 6000,
  "temperature": null,
  "max_attempts": 2,
  "seed": 42,
  "effective_seed": 42,
  "n": null,

Subtype,Requests,Accepted,Acceptance per request
2hop,202,200,99%
3hop1,202,200,99%
3hop2,200,200,100%
4hop1,201,200,100%
4hop2,200,200,100%
4hop3,200,200,100%


Request rejection reasons: {'duplicate': 5}
A successful request can include an earlier rejected model response; inspect provenance for those details.


## Inspect questions and documents

Change `RECORD_INDEX` to select a question (0–29 for a full batch). Inspect all eight criteria below.
Supporting paragraphs and step references use the saved shuffled indices. Distractors and generation history can be expanded.
Passing structural checks does not prove necessary hops or factual consistency.


In [ ]:
def show_record(record_index):
    if not records:
        print("Load a saved batch first.")
        return
    record = records[record_index]
    escape = lambda value: html.escape(str(value))
    parts = [f"<h3>{record_index + 1}/{len(records)} — {escape(record['id'])}</h3>",
             f"<p><b>Question:</b> {escape(record['question'])}</p>",
             f"<p><b>Answer:</b> {escape(record['answer'])} "
             f"<b>Aliases:</b> {escape(record['answer_aliases'])}</p>", "<h4>Decomposition</h4><ol>"]
    for step in record["question_decomposition"]:
        parts.append(f"<li>{escape(step['question'])} → <b>{escape(step['answer'])}</b> "
                     f"(paragraph {step['paragraph_support_idx']})</li>")
    parts.append("</ol><h4>Supporting paragraphs</h4>")
    for paragraph in record["paragraphs"]:
        if paragraph["is_supporting"]:
            parts.append(f"<p><b>[{paragraph['idx']}] {escape(paragraph['title'])}</b><br>"
                         f"{escape(paragraph['paragraph_text'])}</p>")
    parts.append("<details><summary>Distractors</summary>")
    for paragraph in record["paragraphs"]:
        if not paragraph["is_supporting"]:
            parts.append(f"<p><b>[{paragraph['idx']}] {escape(paragraph['title'])}</b><br>"
                         f"{escape(paragraph['paragraph_text'])}</p>")
    parts.append("</details>")
    history = [row for row in provenance_rows if row.get("record_id") == record["id"]]
    parts.append("<details><summary>Generation history and usage</summary><pre>"
                 + escape(json.dumps(history, ensure_ascii=False, indent=2)) + "</pre></details>")
    display(HTML("".join(parts)))


RECORD_INDEX = 0
show_record(RECORD_INDEX)


## Record manual review decisions

Create a CSV template below, then edit it in a spreadsheet or editor. Existing reviews are never overwritten.
Each row includes the record ID and a hash of its exact contents. This is a review worksheet, **not a release exporter**;
`accepted.jsonl` remains the automated-validation output and is not automatically approved for collection.

For each criterion use `pass`, `fail`, or `uncertain`:

| Criterion | Review question |
|---|---|
| answer_support | Do the documents establish one unambiguous answer? |
| internal_consistency | Are the documents and answers mutually consistent? |
| necessary_hops | What necessary information disappears if each supporting paragraph is removed? |
| single_hop_steps | Does each step express one direct lookup? |
| dependency_shape | Are the referenced earlier answers actually needed? |
| no_shortcuts | Can the intended reasoning be bypassed? |
| distractor_quality | Are distractors plausible without contradictions, shortcuts, or alternative answers? |
| question_clarity | Is the main question understandable and sufficiently specific? |

Use `approved`, `rejected`, or `pending` for `decision`; leave uncertain examples pending.
Add your reviewer name and a brief reason. Adoption/retention outcomes are evaluated separately after collection.


In [ ]:
CRITERIA = (
    "answer_support", "internal_consistency", "necessary_hops", "single_hop_steps",
    "dependency_shape", "no_shortcuts", "distractor_quality", "question_clarity",
)
review_path = REVIEW_RUN_DIR / "manual_review.csv"
if not records:
    print("No accepted records loaded; no review file written.")
elif review_path.exists():
    print(f"Keeping existing reviews: {review_path}")
else:
    fields = ["record_id", "candidate_sha256", "hop_type", "decision", "reviewer", *CRITERIA, "notes"]
    with review_path.open("x", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        for record in records:
            canonical = json.dumps(record, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
            writer.writerow({
                "record_id": record["id"],
                "candidate_sha256": hashlib.sha256(canonical.encode("utf-8")).hexdigest(),
                "hop_type": record["id"].split("__", 1)[0],
                "decision": "pending",
            })
    print(f"Review worksheet: {review_path}")


Review worksheet: /home/ubuntu/capstone/fall-2026-group3/data/MuSiQue/synthetic/runs/benchmarks/20261030T172903Z_review30_cafdea1b/manual_review.csv
